In [ ]:
# O mecanismo de atenção (Vaswani et al., 2017) é:
#
#   Attention(Q, K, V) = softmax(QKᵀ / √d_k) @ V
#
# Onde:
#   Q (queries):  "o que cada token está procurando?"
#   K (keys):     "o que cada token oferece?"
#   V (values):   "qual informação cada token carrega?"
#
# QKᵀ é uma MATRIZ DE PRODUTOS INTERNOS:
#   (QKᵀ)_{ij} = qᵢ · kⱼ = similaridade entre token i e token j
#
# Divisão por √d_k: estabilização numérica.
# Sem ela, para d_k grande, os produtos internos ficam grandes,
# o softmax satura, e os gradientes desaparecem.
# Por quê √d_k? Porque se qᵢ e kⱼ têm variância 1, então
# qᵢ·kⱼ tem variância d_k. Dividir por √d_k normaliza para var=1.
#
# softmax: converte scores em pesos de atenção (soma = 1)
#   softmax(z)_i = exp(zᵢ) / Σⱼ exp(zⱼ)
#
# @ V: os valores são somados ponderados pelos pesos de atenção.
#   Cada token produz uma combinação linear dos values de TODOS
#   os outros tokens, ponderada pela atenção.
#
# MULTI-HEAD ATTENTION:
# h cabeças paralelas, cada uma com suas próprias projeções W^Q, W^K, W^V.
# Cada cabeça pode aprender um tipo diferente de relação.
# As saídas são concatenadas e projetadas: W^O @ concat(head_1,...,head_h)
#
# Complexidade: O(n²d) em memória e tempo — o gargalo dos LLMs.
# Flash Attention e variantes resolvem isso via kernel fusion.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

def softmax(x, axis=-1):
    """Softmax numericamente estável (subtrai o max)."""
    x_shifted = x - x.max(axis=axis, keepdims=True)
    ex = np.exp(x_shifted)
    return ex / ex.sum(axis=axis, keepdims=True)

# ============================================================
# PARTE 1 — Atenção escalar: o caso mais simples
# ============================================================
# Antes de matrizes, entenda com 1 query e 3 keys

q = np.array([1., 0., 0., 1.])   # query
K_simples = np.array([
    [1., 0., 0., 1.],   # key 0: similar à query
    [0., 1., 0., 0.],   # key 1: diferente
    [0., 0., 1., 0.],   # key 2: diferente
])
V_simples = np.array([
    [10., 0.],   # value 0
    [0., 10.],   # value 1
    [5.,  5.],   # value 2
])

d_k = K_simples.shape[-1]

scores = K_simples @ q / np.sqrt(d_k)   # qKᵀ (aqui q é 1D)
pesos  = softmax(scores)
saida  = pesos @ V_simples              # combinação linear dos values

print("Scores (produtos internos / √d_k):", scores.round(3))
print("Pesos de atenção (softmax):        ", pesos.round(3))
print("Saída (combinação dos values):     ", saida.round(3))
print("→ A saída está mais próxima do value 0 porque key 0 ≈ query")

# ============================================================
# PARTE 2 — Self-attention em uma sequência
# ============================================================
# Em self-attention, Q = K = V = projeções da mesma sequência.
# Cada token "olha" para todos os outros tokens.

np.random.seed(42)
n_tokens = 6
d_model  = 8    # dimensão dos embeddings
d_k      = 4    # dimensão das projeções Q e K
d_v      = 4    # dimensão das projeções V

# embeddings de entrada simulados (ex: "o gato sentou no tapete")
tokens = ["o", "gato", "sentou", "no", "tapete", "."]
X_seq = np.random.randn(n_tokens, d_model)   # (seq_len, d_model)

# matrizes de projeção (aprendidas durante treino)
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_v) * 0.1

Q = X_seq @ W_Q   # (n_tokens, d_k)
K = X_seq @ W_K   # (n_tokens, d_k)
V = X_seq @ W_V   # (n_tokens, d_v)

# scores: QKᵀ / √d_k — a matriz completa de produtos internos
scores_full = Q @ K.T / np.sqrt(d_k)   # (n_tokens, n_tokens)
atencao     = softmax(scores_full, axis=-1)
saida_seq   = atencao @ V              # (n_tokens, d_v)

print(f"\nSelf-attention:")
print(f"  X_seq:   {X_seq.shape}")
print(f"  Q, K, V: {Q.shape}, {K.shape}, {V.shape}")
print(f"  Scores:  {scores_full.shape}  ← n×n matriz de similaridades")
print(f"  Atenção: {atencao.shape}       ← pesos, soma = 1 por linha")
print(f"  Saída:   {saida_seq.shape}")
print(f"\nSoma de cada linha de atenção:")
print(atencao.sum(axis=1).round(6))   # deve ser exatamente 1

# ============================================================
# PARTE 3 — Masked self-attention (decoder de LLM)
# ============================================================
# Em geração de texto, o token na posição i só pode "ver"
# tokens em posições ≤ i (não o futuro).
# Implementado por uma máscara causal.

mask_causal = np.triu(np.full((n_tokens, n_tokens), -np.inf), k=1)
scores_masked = scores_full + mask_causal
atencao_masked = softmax(scores_masked, axis=-1)

print(f"\nAtenção causal (masked) — linha 2 (token '{tokens[2]}'):")
print(np.round(atencao_masked[2], 3))
print("→ zeros para tokens futuros (posições 3, 4, 5)")

# ============================================================
# PARTE 4 — Multi-head attention
# ============================================================

def single_head_attention(X, W_Q, W_K, W_V, mask=None):
    d_k = W_K.shape[1]
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask
    A = softmax(scores, axis=-1)
    return A @ V, A

def multi_head_attention(X, n_heads, d_model, d_k, d_v, mask=None, seed=0):
    """Multi-head attention com n_heads cabeças paralelas."""
    np.random.seed(seed)
    heads = []
    As = []
    for h in range(n_heads):
        W_Q_h = np.random.randn(d_model, d_k) * 0.1
        W_K_h = np.random.randn(d_model, d_k) * 0.1
        W_V_h = np.random.randn(d_model, d_v) * 0.1
        out_h, A_h = single_head_attention(X, W_Q_h, W_K_h, W_V_h, mask)
        heads.append(out_h)
        As.append(A_h)
    concat = np.concatenate(heads, axis=-1)   # (n_tokens, n_heads * d_v)
    W_O = np.random.randn(n_heads * d_v, d_model) * 0.1
    return concat @ W_O, As

saida_mha, atencoes_heads = multi_head_attention(
    X_seq, n_heads=2, d_model=d_model, d_k=d_k, d_v=d_v)
print(f"\nMulti-head attention (2 cabeças):")
print(f"  Saída: {saida_mha.shape}  ← mesma forma que entrada!")

# ============================================================
# PARTE 5 — Positional encoding
# ============================================================
# Atenção é invariante à ordem (permutação dos tokens = mesmo resultado).
# Positional encoding injeta informação de posição via sinusoides.
#
# PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
# PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
#
# Por que sinusoides? Permitem o modelo aprender deslocamentos
# relativos via adição: PE(pos+k) é combinação linear de PE(pos).

def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, None]
    div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.) / d_model))
    PE[:, 0::2] = np.sin(pos * div)
    PE[:, 1::2] = np.cos(pos * div[:d_model//2])
    return PE

PE = positional_encoding(n_tokens, d_model)
print(f"\nPositional encoding shape: {PE.shape}")
print("Primeiros 4 tokens, 4 primeiras dims:")
print(np.round(PE[:4, :4], 3))

# ============================================================
# PARTE 6 — VISUALIZAÇÃO
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Commit 11 — Atenção como Produto Interno Escalado", fontsize=13)

# Plot 1: heatmap de atenção (self-attention)
ax = axes[0, 0]
im = ax.imshow(atencao, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(n_tokens)); ax.set_xticklabels(tokens, fontsize=9)
ax.set_yticks(range(n_tokens)); ax.set_yticklabels(tokens, fontsize=9)
ax.set_title("Self-attention (bidirec.)\ncada linha = pesos de atenção de 1 token", fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8)

# Plot 2: heatmap de atenção causal
ax2 = axes[0, 1]
im2 = ax2.imshow(atencao_masked, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax2.set_xticks(range(n_tokens)); ax2.set_xticklabels(tokens, fontsize=9)
ax2.set_yticks(range(n_tokens)); ax2.set_yticklabels(tokens, fontsize=9)
ax2.set_title("Atenção causal (decoder)\ntriângulo inferior — não vê o futuro", fontsize=9)
plt.colorbar(im2, ax=ax2, shrink=0.8)

# Plot 3: multi-head — duas cabeças
ax3 = axes[0, 2]
for h, A_h in enumerate(atencoes_heads):
    offset = h * (n_tokens + 1)
fig_mha = axes[0, 2]
im3 = fig_mha.imshow(atencoes_heads[0], cmap='Oranges', vmin=0, vmax=1, aspect='auto')
fig_mha.set_title("Cabeça 1 de 2 (multi-head)\ncada cabeça captura relações diferentes", fontsize=9)
fig_mha.set_xticks(range(n_tokens)); fig_mha.set_xticklabels(tokens, fontsize=9)
fig_mha.set_yticks(range(n_tokens)); fig_mha.set_yticklabels(tokens, fontsize=9)
plt.colorbar(im3, ax=fig_mha, shrink=0.8)

# Plot 4: positional encoding heatmap
ax4 = axes[1, 0]
PE_long = positional_encoding(50, 32)
im4 = ax4.imshow(PE_long.T, cmap='RdBu', aspect='auto')
ax4.set_xlabel("posição no texto")
ax4.set_ylabel("dimensão do embedding")
ax4.set_title("Positional encoding\nsin/cos por dimensão", fontsize=10)
plt.colorbar(im4, ax=ax4, shrink=0.8)

# Plot 5: scores brutos vs após softmax
ax5 = axes[1, 1]
ax5.bar(range(n_tokens), scores_full[2], color='#534AB7', alpha=0.7, label='scores brutos')
ax5_twin = ax5.twinx()
ax5_twin.bar(range(n_tokens), atencao[2], color='#D85A30', alpha=0.5, label='após softmax')
ax5.set_xticks(range(n_tokens)); ax5.set_xticklabels(tokens, fontsize=9)
ax5.set_ylabel("score bruto (QKᵀ/√d)", color='#534AB7')
ax5_twin.set_ylabel("peso de atenção", color='#D85A30')
ax5.set_title(f"Token '{tokens[2]}': scores → pesos", fontsize=10)

# Plot 6: custo quadrático O(n²)
ax6 = axes[1, 2]
seq_lens = np.array([64, 128, 256, 512, 1024, 2048, 4096])
custo = seq_lens**2 / seq_lens[0]**2
ax6.plot(seq_lens, custo, 'o-', color='#D85A30', lw=2, ms=6)
ax6.fill_between(seq_lens, custo, alpha=0.2, color='#D85A30')
ax6.set_xlabel("tamanho da sequência (tokens)")
ax6.set_ylabel("custo relativo (×)")
ax6.set_title("Complexidade O(n²) — gargalo dos LLMs", fontsize=10)
ax6.set_yscale('log')
ax6.grid(True, alpha=0.3)
for sl, c in zip(seq_lens, custo):
    ax6.annotate(f'{c:.0f}×', (sl, c), textcoords='offset points',
                 xytext=(4, 4), fontsize=8)

plt.tight_layout()
plt.savefig('assets/11_attention.png', dpi=150, bbox_inches='tight')
plt.show()